In [1]:
# ============================================================
# PROJECT 1: Customer Churn Prediction
# Phase 4: Power BI Data Preparation
# ============================================================

import pandas as pd
import numpy as np

df = pd.read_csv('telco_scored.csv')

# Rebuild categorical columns cleanly
df['value_tier'] = pd.qcut(df['cltv'], q=3,
                            labels=['Low Value', 'Mid Value', 'High Value'])
df['churn_risk_tier'] = pd.cut(df['churn_probability'],
                                bins=[0, 0.3, 0.6, 1.0],
                                labels=['Low Risk', 'Medium Risk', 'High Risk'])
df['tenure_band'] = pd.cut(df['tenure_months'],
                            bins=[0, 6, 12, 24, 48, 72],
                            labels=['0-6 mo', '7-12 mo', '13-24 mo',
                                    '25-48 mo', '49+ mo'])

# Convert to string so Power BI reads them as text categories
df['value_tier']      = df['value_tier'].astype(str)
df['churn_risk_tier'] = df['churn_risk_tier'].astype(str)
df['tenure_band']     = df['tenure_band'].astype(str)

print("Base data ready:", df.shape)

Base data ready: (7043, 13)


In [2]:
# ── TABLE 1: KPI Summary (single row — for KPI cards) ─────────
kpi_summary = pd.DataFrame([{
    'total_customers'       : len(df),
    'total_churners'        : df['churn_value'].sum(),
    'overall_churn_rate'    : round(df['churn_value'].mean() * 100, 1),
    'monthly_rev_at_risk'   : round(df[df['churn_value']==1]['monthly_charges'].sum(), 0),
    'annual_rev_at_risk'    : round(df[df['churn_value']==1]['monthly_charges'].sum() * 12, 0),
    'avg_cltv_churner'      : round(df[df['churn_value']==1]['cltv'].mean(), 0),
    'avg_cltv_retained'     : round(df[df['churn_value']==0]['cltv'].mean(), 0),
    'high_risk_customers'   : len(df[df['churn_risk_tier']=='High Risk']),
}])

kpi_summary.to_csv('pbi_kpi_summary.csv', index=False)
print("✓ pbi_kpi_summary.csv")

✓ pbi_kpi_summary.csv


In [3]:
# ── TABLE 2: Churn by Value Tier (bar + donut charts) ─────────
churn_by_tier = df.groupby('value_tier').agg(
    total_customers     = ('churn_value', 'count'),
    churners            = ('churn_value', 'sum'),
    churn_rate_pct      = ('churn_value', lambda x: round(x.mean()*100, 1)),
    avg_monthly_charges = ('monthly_charges', 'mean'),
    avg_cltv            = ('cltv', 'mean'),
    monthly_rev_at_risk = ('monthly_charges', lambda x: round(
                            x[df.loc[x.index,'churn_value']==1].sum(), 0)),
).reset_index()

churn_by_tier['annual_rev_at_risk'] = churn_by_tier['monthly_rev_at_risk'] * 12
churn_by_tier['avg_monthly_charges'] = churn_by_tier['avg_monthly_charges'].round(2)
churn_by_tier['avg_cltv'] = churn_by_tier['avg_cltv'].round(0)

# Sort order for Power BI visuals
tier_order = {'Low Value': 1, 'Mid Value': 2, 'High Value': 3}
churn_by_tier['sort_order'] = churn_by_tier['value_tier'].map(tier_order)
churn_by_tier = churn_by_tier.sort_values('sort_order')

churn_by_tier.to_csv('pbi_churn_by_tier.csv', index=False)
print("✓ pbi_churn_by_tier.csv")

✓ pbi_churn_by_tier.csv


In [4]:
# ── TABLE 3: Risk Matrix (heatmap — the centrepiece visual) ───
risk_matrix = df.groupby(['value_tier', 'churn_risk_tier']).agg(
    customers           = ('churn_value', 'count'),
    churners            = ('churn_value', 'sum'),
    churn_rate_pct      = ('churn_value', lambda x: round(x.mean()*100,1)),
    annual_rev_at_risk  = ('monthly_charges', lambda x: round(
                            x[df.loc[x.index,'churn_value']==1].sum()*12, 0)),
    avg_churn_prob      = ('churn_probability', lambda x: round(x.mean()*100, 1))
).reset_index()

risk_matrix.to_csv('pbi_risk_matrix.csv', index=False)
print("✓ pbi_risk_matrix.csv")

✓ pbi_risk_matrix.csv


In [5]:
# ── TABLE 4: Churn by Contract Type ───────────────────────────
churn_by_contract = df.groupby('contract').agg(
    total_customers = ('churn_value', 'count'),
    churners        = ('churn_value', 'sum'),
    churn_rate_pct  = ('churn_value', lambda x: round(x.mean()*100,1)),
    avg_cltv        = ('cltv', lambda x: round(x.mean(),0))
).reset_index()

churn_by_contract.to_csv('pbi_churn_by_contract.csv', index=False)
print("✓ pbi_churn_by_contract.csv")

✓ pbi_churn_by_contract.csv


In [6]:
# ── TABLE 5: Churn by Tenure Band ─────────────────────────────
tenure_order = {'0-6 mo':1,'7-12 mo':2,'13-24 mo':3,'25-48 mo':4,'49+ mo':5}

churn_by_tenure = df.groupby('tenure_band').agg(
    total_customers = ('churn_value', 'count'),
    churners        = ('churn_value', 'sum'),
    churn_rate_pct  = ('churn_value', lambda x: round(x.mean()*100,1)),
).reset_index()

churn_by_tenure['sort_order'] = churn_by_tenure['tenure_band'].map(tenure_order)
churn_by_tenure = churn_by_tenure.sort_values('sort_order')

churn_by_tenure.to_csv('pbi_churn_by_tenure.csv', index=False)
print("✓ pbi_churn_by_tenure.csv")

✓ pbi_churn_by_tenure.csv


In [7]:
# ── TABLE 6: Churn Reasons (bar chart) ────────────────────────
if 'churn_reason' in df.columns:
    churn_reasons = (df[df['churn_value']==1]['churn_reason']
                     .dropna()
                     .value_counts()
                     .reset_index())
    churn_reasons.columns = ['reason', 'count']
    churn_reasons['pct'] = (churn_reasons['count'] /
                             churn_reasons['count'].sum() * 100).round(1)
    churn_reasons = churn_reasons.head(10)
    churn_reasons.to_csv('pbi_churn_reasons.csv', index=False)
    print("✓ pbi_churn_reasons.csv")

✓ pbi_churn_reasons.csv


In [8]:
# ── TABLE 7: Retention ROI Table ──────────────────────────────
SAVE_RATE          = 0.40
RETENTION_COST_PER = 50  # $ per customer

retention_roi = []
for (value, risk), group in df.groupby(['value_tier','churn_risk_tier']):
    n               = len(group)
    annual_rev      = group['monthly_charges'].sum() * 12
    saveable_rev    = annual_rev * SAVE_RATE
    spend           = n * RETENTION_COST_PER
    roi             = ((saveable_rev - spend) / spend * 100) if spend > 0 else 0
    retention_roi.append({
        'value_tier'       : value,
        'churn_risk_tier'  : risk,
        'customers'        : n,
        'annual_rev_at_risk': round(annual_rev, 0),
        'saveable_revenue' : round(saveable_rev, 0),
        'retention_spend'  : round(spend, 0),
        'roi_pct'          : round(roi, 1),
        'recommendation'   : ('✅ PRIORITISE' if risk=='High Risk' and value=='High Value'
                              else '⚠ CONSIDER'  if risk=='High Risk' and value=='Mid Value'
                              else '❌ SKIP'      if value=='Low Value'
                              else '— MONITOR')
    })

retention_df = pd.DataFrame(retention_roi)
retention_df.to_csv('pbi_retention_roi.csv', index=False)
print("✓ pbi_retention_roi.csv")

✓ pbi_retention_roi.csv


In [10]:
# ── TABLE 8: Customer-level detail (for drill-through) ────────
detail_cols = ['customer_id','monthly_charges','total_charges','cltv',
               'tenure_months','contract','churn_value','churn_probability',
               'churn_risk_tier','value_tier','churn_reason']
detail_cols = [c for c in detail_cols if c in df.columns]

df[detail_cols].to_csv('pbi_customer_detail.csv', index=False)
print("✓ pbi_customer_detail.csv")

print("\n✅ All Power BI tables exported. Move all CSV files to one folder.")

✓ pbi_customer_detail.csv

✅ All Power BI tables exported. Move all CSV files to one folder.


In [11]:
churn_by_tier['annual_rev_at_risk'] = churn_by_tier['annual_rev_at_risk'].astype(float)
churn_by_tier['churn_rate_pct']     = churn_by_tier['churn_rate_pct'].astype(float)
churn_by_tier['total_customers']    = churn_by_tier['total_customers'].astype(int)
churn_by_tier['churners']           = churn_by_tier['churners'].astype(int)
churn_by_tier['sort_order']         = churn_by_tier['sort_order'].astype(int)

churn_by_tier.to_csv('pbi_churn_by_tier.csv', index=False)